# v16.2 — uji ambang t = −1 detektor AF 2 kelas (ECGFounder) di CPSC 2021

Rancangan dikunci sebelum sinyal CPSC 2021 dibaca: [RANCANGAN.md](RANCANGAN.md) (sha256 di `../logs/v16.2_praregistrasi.log`).
Logika di `uji_cpsc.py`. CPSC 2021 belum pernah dinilai oleh model SIHEDAF mana pun; tidak ada yang dilatih atau disetel di sini.

| Sistem | Keputusan AF |
|---|---|
| **D-1 (primer)** | ECGFounder zero-shot: log((pAF + pAFL) / pN) > −1 |
| D-0 (v16.1) | ... > 0 |
| **v3.1 paket (referensi)**, v1 paket | argmax paket pada RR window, AFIB + AFL digabung |

In [1]:
import json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import uji_cpsc as U
from uji_cpsc import A161, D, DK, E13, Ev, H16, K33


def dl(v):
    return f"Δ {v[0]:+.3f} [{v[1]:+.3f}…{v[2]:+.3f}], P = {v[3]:.3f} → {Ev.vonis(v[3])}"


J = U.bangun_jendela()
nama = {0: "Normal", 1: "AFIB", 2: "AFL"}
ringkas_data = {nama[c]: dict(window=int((J["y"] == c).sum()), lolos=int(((J["y"] == c) & J["ok"]).sum()),
                              subjek=int(np.unique(J["subjek"][(J["y"] == c) & J["ok"]]).size)) for c in (0, 1, 2)}
print("rekaman dengan window berlabel:", np.unique(J["rekaman"]).size, "| subjek:", np.unique(J["subjek"]).size)
pd.DataFrame(ringkas_data).T.rename_axis("CPSC 2021 (window 10 s, langkah 5 s)")

rekaman dengan window berlabel: 700 | subjek: 54


,window,lolos,subjek
"CPSC 2021 (window 10 s, langkah 5 s)",,,
Normal,13645,13495,22
AFIB,116866,111320,53
AFL,144,144,2


In [2]:
ok = J["ok"]
lo = U.logit_ecgfounder(J["ecg"][ok])
_, d1 = H16.tahap1(lo)
p31, v1 = U.paket(J["X"][ok], J["M"][ok])
y, subj, rek, t0, hr = J["y"][ok], J["subjek"][ok], J["rekaman"][ok], J["start"][ok], J["hr"][ok]
y2 = (y > 0).astype(np.int64)
pred = {"v3.1 paket": (p31.argmax(1) != 0).astype(np.int64), "v1 paket": (v1 != 0).astype(np.int64),
        "D-0": (d1 > 0).astype(np.int64), "D-1": (d1 > U.AMBANG).astype(np.int64)}
print("window uji:", len(y), "| AF:", int(y2.sum()), "| Normal:", int((y2 == 0).sum()), "| subjek:", np.unique(subj).size)

window uji: 124959 | AF: 111464 | Normal: 13495 | subjek: 54


## 1. Hasil utama (H1–H3): bootstrap subjek berpasangan 2.000×

In [3]:
bd = {k: A161.banding2_pasien(y2, pred[k], pred["v3.1 paket"], subj) for k in ("D-1", "D-0", "v1 paket")}
bd["D-1 vs D-0"] = A161.banding2_pasien(y2, pred["D-1"], pred["D-0"], subj)
titik = {"v3.1 paket": bd["D-1"]["titik_b"], **{k: bd[k]["titik_a"] for k in ("D-1", "D-0", "v1 paket")}}
display(pd.DataFrame({k: {m: f"{v[m][0]:.3f} [{v[m][1]:.2f}–{v[m][2]:.2f}]" for m in ("Macro-F1", "Se", "Sp", "PPV")} for k, v in titik.items()}).T
        .rename_axis("CPSC 2021, 2 kelas"))
pd.DataFrame({k: {m: dl(v[m]) for m in ("Macro-F1", "Se", "Sp", "PPV")} for k, v in bd.items()}).T.rename_axis("vs v3.1 paket (kecuali baris terakhir)")

,Macro-F1,Se,Sp,PPV
"CPSC 2021, 2 kelas",,,,
v3.1 paket,0.883 [0.78–0.93],0.951 [0.93–0.97],0.926 [0.88–0.96],0.991 [0.98–1.00]
D-1,0.908 [0.81–0.95],0.955 [0.94–0.97],0.994 [0.98–1.00],0.999 [1.00–1.00]
D-0,0.835 [0.70–0.91],0.906 [0.87–0.94],0.998 [0.99–1.00],1.000 [1.00–1.00]
v1 paket,0.892 [0.79–0.93],0.957 [0.94–0.97],0.925 [0.88–0.96],0.991 [0.98–1.00]


,Macro-F1,Se,Sp,PPV
vs v3.1 paket (kecuali baris terakhir),,,,
D-1,"Δ +0.025 [-0.003…+0.055], P = 0.964 → lebih baik","Δ +0.003 [-0.012…+0.017], P = 0.688 → tidak te...","Δ +0.068 [+0.033…+0.110], P = 1.000 → lebih baik","Δ +0.009 [+0.002…+0.019], P = 1.000 → lebih baik"
D-0,"Δ -0.048 [-0.095…-0.011], P = 0.004 → lebih buruk","Δ -0.045 [-0.074…-0.023], P = 0.000 → lebih buruk","Δ +0.072 [+0.035…+0.120], P = 1.000 → lebih baik","Δ +0.009 [+0.003…+0.020], P = 1.000 → lebih baik"
v1 paket,"Δ +0.009 [-0.004…+0.026], P = 0.904 → tidak te...","Δ +0.005 [-0.002…+0.014], P = 0.921 → tidak te...","Δ -0.001 [-0.009…+0.014], P = 0.429 → tidak te...","Δ -0.000 [-0.001…+0.001], P = 0.461 → tidak te..."
D-1 vs D-0,"Δ +0.073 [+0.043…+0.102], P = 1.000 → lebih baik","Δ +0.049 [+0.034…+0.067], P = 1.000 → lebih baik","Δ -0.004 [-0.011…-0.001], P = 0.000 → lebih buruk","Δ -0.001 [-0.001…-0.000], P = 0.001 → lebih buruk"


In [4]:
per_kelas = {k: {nama[c]: float(p[y == c].mean()) for c in (0, 1, 2)} for k, p in pred.items()}
display(pd.DataFrame(per_kelas).T.round(3).rename_axis("proporsi alarm AF per kelas asli"))
pita = {k: U.alarm_pita(y2, p, hr) for k, p in pred.items()}
display(pd.DataFrame({k: {b: (f"{x['alarm']*100:.1f}% (n={x['n']})" if x["alarm"] is not None else "–") for b, x in v.items()} for k, v in pita.items()}).T
        .rename_axis("alarm pada Normal per pita detak"))
ps = {k: U.per_subjek(y2, pred[k], subj) for k in ("v3.1 paket", "D-0", "D-1")}
turun = {s: round(ps["v3.1 paket"][s]["se"] - ps["D-1"][s]["se"], 3) for s in ps["D-1"]
         if ps["D-1"][s]["se"] is not None and ps["v3.1 paket"][s]["se"] - ps["D-1"][s]["se"] > 0.20}
print("subjek dengan sensitivitas D-1 turun > 0,20 dari v3.1:", turun, "dari", sum(v["se"] is not None for v in ps["D-1"].values()), "subjek ber-AF")
ep = {k: K33.metrik_episode(rek, t0, y2, p, 1, langkah=5.0) for k, p in pred.items()}
pd.DataFrame(ep).T.round(3).rename_axis("episode AF (EC57, window yang lolos cek)")

,Normal,AFIB,AFL
proporsi alarm AF per kelas asli,,,
v3.1 paket,0.074,0.951,0.979
v1 paket,0.075,0.957,1.000
D-0,0.002,0.906,0.882
D-1,0.006,0.955,0.917


,< 100,100–110,110–120,≥ 120
alarm pada Normal per pita detak,,,,
v3.1 paket,7.1% (n=13433),64.9% (n=57),100.0% (n=5),–
v1 paket,7.2% (n=13433),63.2% (n=57),100.0% (n=5),–
D-0,0.1% (n=13433),3.5% (n=57),40.0% (n=5),–
D-1,0.5% (n=13433),7.0% (n=57),60.0% (n=5),–


subjek dengan sensitivitas D-1 turun > 0,20 dari v3.1: {'61': 0.333} dari 54 subjek ber-AF


,se_episode,ppv_episode,se_durasi,ppv_durasi,n_ref,n_det,palsu_per_jam
"episode AF (EC57, window yang lolos cek)",,,,,,,
v3.1 paket,0.959,0.932,0.951,0.991,2956.0,6455.0,2.541
v1 paket,0.939,0.911,0.957,0.991,2956.0,5653.0,2.887
D-0,0.926,0.998,0.906,1.000,2956.0,8501.0,0.104
D-1,0.974,0.990,0.955,0.999,2956.0,6093.0,0.351


## 1b. Label resmi CPSC 2021 (penyimpangan tercatat, RANCANGAN bagian 8)

Konvensi resmi (`score_2021.py`): `(AFIB`/`(AFL` = awal episode AF, `(N` = akhir; rekaman "non atrial fibrillation" tanpa label ritme.
Bagian 1 memakai aturan yang dikunci, yang membuang 721 rekaman non-AF. Di sini: AF bila window di dalam interval AF, Normal bila tidak
bersinggungan dengan interval AF, dibuang bila melintasi batas. Sistem, ambang, metrik, dan kriteria sama. Window AF di-assert identik.

In [5]:
Jr = U.bangun_jendela(resmi=True)
okr = Jr["ok"]
data_r = {nama[c]: dict(window=int((Jr["y"] == c).sum()), lolos=int(((Jr["y"] == c) & okr).sum()),
                        subjek=int(np.unique(Jr["subjek"][(Jr["y"] == c) & okr]).size)) for c in (0, 1, 2)}
kunci = lambda Z, m: set(zip(Z["rekaman"][m], np.round(Z["start"][m], 3), Z["y"][m]))  # noqa: E731
assert kunci(J, J["y"] > 0) == kunci(Jr, Jr["y"] > 0), "window AF berbeda antar aturan label"
assert kunci(J, J["y"] == 0) <= kunci(Jr, Jr["y"] == 0)
lor = U.logit_ecgfounder(Jr["ecg"][okr], "logit_ecgfounder_resmi")
_, d1r = H16.tahap1(lor)
p31r, v1r = U.paket(Jr["X"][okr], Jr["M"][okr], "paket_resmi")
yr, subjr, rekr, t0r, hrr = (Jr[k][okr] for k in ("y", "subjek", "rekaman", "start", "hr"))
# window AF yang sama di kedua run: prediksi paket harus identik, keputusan ECGFounder dicatat (batch GPU berbeda)
pos = {kk: i for i, kk in enumerate(zip(rek, np.round(t0, 3)))}
af_r = np.flatnonzero(yr > 0)
idx_lama = np.array([pos[kk] for kk in zip(rekr[af_r], np.round(t0r[af_r], 3))])
assert np.allclose(p31r[af_r], p31[idx_lama], atol=1e-6) and np.array_equal(v1r[af_r], v1[idx_lama])
cek_af = dict(n=int(len(af_r)), keputusan_d1_sama=float(np.mean((d1r[af_r] > U.AMBANG) == (d1[idx_lama] > U.AMBANG))),
              maks_selisih_d1=float(np.abs(d1r[af_r] - d1[idx_lama]).max()))
print("window AF di kedua run:", cek_af)
y2r = (yr > 0).astype(np.int64)
predr = {"v3.1 paket": (p31r.argmax(1) != 0).astype(np.int64), "v1 paket": (v1r != 0).astype(np.int64),
         "D-0": (d1r > 0).astype(np.int64), "D-1": (d1r > U.AMBANG).astype(np.int64)}
bdr = {k: A161.banding2_pasien(y2r, predr[k], predr["v3.1 paket"], subjr) for k in ("D-1", "D-0", "v1 paket")}
bdr["D-1 vs D-0"] = A161.banding2_pasien(y2r, predr["D-1"], predr["D-0"], subjr)
titik_r = {"v3.1 paket": bdr["D-1"]["titik_b"], **{k: bdr[k]["titik_a"] for k in ("D-1", "D-0", "v1 paket")}}
per_kelas_r = {k: {nama[c]: float(p[yr == c].mean()) for c in (0, 1, 2)} for k, p in predr.items()}
pita_r = {k: U.alarm_pita(y2r, p, hrr) for k, p in predr.items()}
ps_r = {k: U.per_subjek(y2r, predr[k], subjr) for k in ("v3.1 paket", "D-0", "D-1")}
turun_r = {s: round(ps_r["v3.1 paket"][s]["se"] - ps_r["D-1"][s]["se"], 3) for s in ps_r["D-1"]
           if ps_r["D-1"][s]["se"] is not None and ps_r["v3.1 paket"][s]["se"] - ps_r["D-1"][s]["se"] > 0.20}
ep_r = {k: K33.metrik_episode(rekr, t0r, y2r, p, 1, langkah=5.0) for k, p in predr.items()}
print("rekaman:", np.unique(Jr["rekaman"]).size, "| subjek:", np.unique(Jr["subjek"]).size, "| window uji:", len(yr), "| AF:", int(y2r.sum()),
      "| Normal:", int((y2r == 0).sum()), "| subjek uji:", np.unique(subjr).size)
display(pd.DataFrame(data_r).T.rename_axis("label resmi"))
display(pd.DataFrame({k: {m: f"{v[m][0]:.3f} [{v[m][1]:.2f}–{v[m][2]:.2f}]" for m in ("Macro-F1", "Se", "Sp", "PPV")} for k, v in titik_r.items()}).T
        .rename_axis("CPSC 2021 label resmi, 2 kelas"))
display(pd.DataFrame({k: {m: dl(v[m]) for m in ("Macro-F1", "Se", "Sp", "PPV")} for k, v in bdr.items()}).T.rename_axis("label resmi: vs v3.1 paket"))
display(pd.DataFrame(per_kelas_r).T.round(3).rename_axis("label resmi: alarm per kelas asli"))
display(pd.DataFrame({k: {b: (f"{x['alarm']*100:.1f}% (n={x['n']})" if x["alarm"] is not None else "–") for b, x in v.items()} for k, v in pita_r.items()}).T
        .rename_axis("label resmi: alarm Normal per pita"))
print("label resmi: subjek dengan Se D-1 turun > 0,20:", turun_r)
pd.DataFrame(ep_r).T.round(3).rename_axis("label resmi: episode AF")

window AF di kedua run: {'n': 111464, 'keputusan_d1_sama': 1.0, 'maks_selisih_d1': 0.03318332668869761}


rekaman: 1424 | subjek: 105 | window uji: 321818 | AF: 111464 | Normal: 210354 | subjek uji: 105


,window,lolos,subjek
label resmi,,,
Normal,224430,210354,76
AFIB,116866,111320,53
AFL,144,144,2


,Macro-F1,Se,Sp,PPV
"CPSC 2021 label resmi, 2 kelas",,,,
v3.1 paket,0.881 [0.83–0.91],0.951 [0.93–0.97],0.853 [0.80–0.90],0.774 [0.65–0.86]
D-1,0.953 [0.92–0.97],0.955 [0.94–0.97],0.958 [0.92–0.98],0.923 [0.84–0.97]
D-0,0.946 [0.92–0.96],0.906 [0.87–0.94],0.976 [0.95–0.99],0.951 [0.89–0.99]
v1 paket,0.867 [0.82–0.90],0.957 [0.94–0.97],0.829 [0.77–0.87],0.748 [0.61–0.84]


,Macro-F1,Se,Sp,PPV
label resmi: vs v3.1 paket,,,,
D-1,"Δ +0.072 [+0.047…+0.106], P = 1.000 → lebih baik","Δ +0.003 [-0.011…+0.017], P = 0.697 → tidak te...","Δ +0.105 [+0.072…+0.142], P = 1.000 → lebih baik","Δ +0.149 [+0.091…+0.229], P = 1.000 → lebih baik"
D-0,"Δ +0.065 [+0.036…+0.105], P = 1.000 → lebih baik","Δ -0.045 [-0.076…-0.023], P = 0.000 → lebih buruk","Δ +0.123 [+0.089…+0.162], P = 1.000 → lebih baik","Δ +0.178 [+0.111…+0.272], P = 1.000 → lebih baik"
v1 paket,"Δ -0.013 [-0.023…-0.005], P = 0.001 → lebih buruk","Δ +0.005 [-0.002…+0.013], P = 0.929 → tidak te...","Δ -0.024 [-0.036…-0.012], P = 0.000 → lebih buruk","Δ -0.026 [-0.043…-0.012], P = 0.000 → lebih buruk"
D-1 vs D-0,"Δ +0.007 [-0.006…+0.018], P = 0.860 → tidak te...","Δ +0.049 [+0.034…+0.068], P = 1.000 → lebih baik","Δ -0.018 [-0.029…-0.008], P = 0.000 → lebih buruk","Δ -0.028 [-0.053…-0.011], P = 0.000 → lebih buruk"


,Normal,AFIB,AFL
label resmi: alarm per kelas asli,,,
v3.1 paket,0.147,0.951,0.979
v1 paket,0.171,0.957,1.000
D-0,0.024,0.906,0.882
D-1,0.042,0.955,0.917


,< 100,100–110,110–120,≥ 120
label resmi: alarm Normal per pita,,,,
v3.1 paket,14.2% (n=203734),18.2% (n=4587),48.9% (n=1579),93.8% (n=454)
v1 paket,16.3% (n=203734),27.8% (n=4587),73.2% (n=1579),86.6% (n=454)
D-0,2.4% (n=203734),3.2% (n=4587),4.6% (n=1579),18.3% (n=454)
D-1,4.1% (n=203734),5.3% (n=4587),7.0% (n=1579),25.1% (n=454)


label resmi: subjek dengan Se D-1 turun > 0,20: {'61': 0.333}


,se_episode,ppv_episode,se_durasi,ppv_durasi,n_ref,n_det,palsu_per_jam
label resmi: episode AF,,,,,,,
v3.1 paket,0.959,0.343,0.951,0.774,2956.0,17510.0,25.720
v1 paket,0.939,0.278,0.957,0.748,2956.0,18518.0,29.904
D-0,0.926,0.812,0.906,0.951,2956.0,10452.0,4.405
D-1,0.974,0.658,0.955,0.923,2956.0,9174.0,7.030


## 2. H4 (sekunder): D-1 di data yang sudah dipakai untuk melihat kurva v16.1

AFDB (aturan 2 kelas v11.1), PTB-XL, Icentia11k, dan sinus takikardia asli. Ambang −1 dipilih setelah melihat titik-titik ini, jadi bagian ini bukan uji independen.

In [6]:
d = D.muat()
R = len(d.folds)
y2a = (d.y > 0).astype(np.int64)
d1a = A161.d1_afdb()
W = A161.cv.bootstrap_weights(d.strata, Ev.N_BOOT, seed=D.SEED)
p31a = A161.ke2(Ev.keputusan(d.oof_v31, d.folds, d.y)[0])
afdb = {}
bootA = {}
for k, p in (("v3.1", p31a), ("D-1", np.repeat((d1a > U.AMBANG).astype(np.int64)[None], R, 0)), ("D-0", np.repeat((d1a > 0).astype(np.int64)[None], R, 0))):
    afdb[k], bootA[k] = DK.ringkas(DK.cm_pasien(y2a, p, d.records, d.patients), W)
afdb_bd = {k: DK.banding(afdb[k], afdb["v3.1"], bootA[k], bootA["v3.1"]) for k in ("D-1", "D-0")}
r161 = json.loads((D.REPO / "cache/v16.1/ringkasan.json").read_text())
assert np.isclose(afdb["v3.1"]["Macro-F1"][0], r161["afdb"]["v3.1"]["Macro-F1"][0]) and np.isclose(afdb["D-0"]["Macro-F1"][0], r161["afdb"]["D-0"]["Macro-F1"][0])
eks = {}
P = H16.ptbxl(); _, d1p = H16.tahap1(P["logit"])
eks["PTB-XL"] = A161.banding2_pasien((P["y"] > 0).astype(np.int64), d1p > U.AMBANG, P["p31"].argmax(1) != 0, P["pasien"])
I = H16.icentia(); _, d1i = H16.tahap1(I["logit"])
eks["Icentia11k"] = A161.banding2_pasien((I["y"] > 0).astype(np.int64), d1i > U.AMBANG, I["p31"].argmax(1) != 0, I["pasien"])
L = H16.latihan(); Jl = L["J"]; _, d1l = H16.tahap1(L["logit"])
kl = {"v3.1 paket": H16.kelas_latihan(L, (L["p31"].argmax(1) != 0).astype(np.int64)), "D-1": H16.kelas_latihan(L, (d1l > U.AMBANG).astype(np.int64))}
hr_m, subj_m = (Jl[k][Jl["ok"]][L["mask"]] for k in ("hr", "subjek"))
m120 = hr_m >= 120
lat = {}
for k, v in kl.items():
    a = (v[:, L["mask"]] != 0).mean(0)
    t, ci, _ = K33._boot_subjek(subj_m[m120], a[m120])
    lat[k] = dict(alarm=t, ci=ci.tolist(), pita=K33.alarm_per_pita(Jl, v, L["mask"]))
lat["D-1"]["banding"] = K33.banding_alarm(Jl, kl["D-1"], kl["v3.1 paket"], L["mask"])
display(pd.DataFrame({k: {m: dl(v[m]) for m in ("Macro-F1", "Se", "Sp")} for k, v in {"AFDB D-1": afdb_bd["D-1"], **{f"{n} D-1": v for n, v in eks.items()}}.items()}).T
        .rename_axis("D-1 vs v3.1 (data yang sudah dipakai)"))
print("AFDB D-1:", {m: round(afdb["D-1"][m][0], 3) for m in ("Macro-F1", "Se", "Sp")}, "| v3.1:", {m: round(afdb["v3.1"][m][0], 3) for m in ("Macro-F1", "Se", "Sp")})
print("sinus takikardia asli ≥120 bpm: D-1", round(lat["D-1"]["alarm"] * 100, 2), "%", np.round(np.array(lat["D-1"]["ci"]) * 100, 2),
      "| v3.1", round(lat["v3.1 paket"]["alarm"] * 100, 1), "%")

,Macro-F1,Se,Sp
D-1 vs v3.1 (data yang sudah dipakai),,,
AFDB D-1,"Δ +0.042 [+0.013…+0.084], P = 1.000 → lebih baik","Δ +0.002 [-0.029…+0.024], P = 0.573 → tidak te...","Δ +0.068 [+0.026…+0.147], P = 1.000 → lebih baik"
PTB-XL D-1,"Δ +0.122 [+0.112…+0.131], P = 1.000 → lebih baik","Δ -0.005 [-0.023…+0.012], P = 0.268 → tidak te...","Δ +0.053 [+0.050…+0.057], P = 1.000 → lebih baik"
Icentia11k D-1,"Δ +0.012 [+0.001…+0.024], P = 0.979 → lebih baik","Δ -0.029 [-0.044…-0.013], P = 0.000 → lebih buruk","Δ +0.089 [+0.080…+0.098], P = 1.000 → lebih baik"


AFDB D-1: {'Macro-F1': 0.968, 'Se': 0.935, 'Sp': 0.992} | v3.1: {'Macro-F1': 0.926, 'Se': 0.934, 'Sp': 0.924}
sinus takikardia asli ≥120 bpm: D-1 0.46 % [0.1  0.94] | v3.1 98.4 %


In [7]:
kriteria = dict(H1a_macro_f1=bool(bd["D-1"]["Macro-F1"][3] > 0.05), H1b_sensitivitas=bool(bd["D-1"]["Se"][3] > 0.05),
                sinus_takikardia=bool(lat["D-1"]["alarm"] <= 0.5 * lat["v3.1 paket"]["alarm"]))
kriteria_resmi = dict(H1a_macro_f1=bool(bdr["D-1"]["Macro-F1"][3] > 0.05), H1b_sensitivitas=bool(bdr["D-1"]["Se"][3] > 0.05),
                      sinus_takikardia=kriteria["sinus_takikardia"])
kriteria["kandidat_uji_perangkat"] = all(kriteria.values()) and all(kriteria_resmi.values())
ring = {"data": ringkas_data, "n_uji": dict(window=int(len(y)), af=int(y2.sum()), normal=int((y2 == 0).sum()), subjek=int(np.unique(subj).size),
                                             rekaman=int(np.unique(rek).size)),
        "banding": bd, "titik": titik, "per_kelas": per_kelas, "alarm_pita": pita, "per_subjek": ps, "se_turun_0_20": turun, "episode": ep,
        "h4": {"afdb": afdb, "afdb_banding": afdb_bd, "eksternal": eks, "latihan": lat}, "kriteria": kriteria, "ambang": U.AMBANG,
        "label_resmi": {"data": data_r, "n_uji": dict(window=int(len(yr)), af=int(y2r.sum()), normal=int((y2r == 0).sum()),
                                                     subjek=int(np.unique(subjr).size), rekaman=int(np.unique(rekr).size)),
                        "banding": bdr, "titik": titik_r, "per_kelas": per_kelas_r, "alarm_pita": pita_r, "per_subjek": ps_r,
                        "se_turun_0_20": turun_r, "episode": ep_r, "cek_af": cek_af, "kriteria": kriteria_resmi},
        "lingkungan": Ev._lingkungan()}
(U.CACHE / "ringkasan.json").write_text(json.dumps(ring, indent=1, default=float, ensure_ascii=False))
print("kriteria (dikunci):", kriteria)
print("kriteria (label resmi):", kriteria_resmi)
print("tersimpan:", U.CACHE / "ringkasan.json")

kriteria (dikunci): {'H1a_macro_f1': True, 'H1b_sensitivitas': True, 'sinus_takikardia': True, 'kandidat_uji_perangkat': True}
kriteria (label resmi): {'H1a_macro_f1': True, 'H1b_sensitivitas': True, 'sinus_takikardia': True}
tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v16.2/ringkasan.json


## 3. Kesimpulan

Dijalankan penuh 6 November 2026 di PC (RTX 5060 untuk ECGFounder; ±10 menit termasuk membaca 1.425 rekaman). Kriteria yang dikunci lolos
**di kedua analisis** (aturan label yang dikunci dan label resmi CPSC 2021) → **D-1 (ECGFounder, AF bila d1 > −1) adalah kandidat
detektor AF 2 kelas untuk uji perangkat.** Bukan pengganti paket v3.1 dan tidak dikemas.

- **Analisis yang dikunci** (window Normal hanya dari 22 subjek, lihat penyimpangan): Macro-F1 2 kelas D-1 0,908 vs v3.1 0,883
  (Δ +0,025, P = 0,964, lebih baik); sensitivitas 0,955 vs 0,951 (P = 0,69, tidak terbedakan); spesifisitas 0,994 vs 0,926 (lebih baik).
- **Label resmi CPSC 2021** (1.424 rekaman, 105 subjek, 321.818 window; window AF identik dengan analisis pertama): D-1 0,953 vs v3.1
  0,881 (Δ +0,072, P = 1,00, lebih baik); sensitivitas 0,955 vs 0,951 (P = 0,70); spesifisitas 0,958 vs 0,853; PPV 0,923 vs 0,774. Episode
  AF palsu 25,7 → 7,0 per jam dengan Se episode 0,96 → 0,97.
- **Ambang −1 lebih baik dari 0 di data baru:** D-0 kehilangan sensitivitas (0,906, lebih buruk dari v3.1) dan Macro-F1-nya lebih buruk
  pada analisis yang dikunci (0,835); D-1 − D-0 sensitivitas +0,049 (P = 1,00).
- **Data yang sudah dipakai (H4, bukan uji independen):** AFDB 0,968 vs 0,926, PTB-XL 0,939 vs 0,817, Icentia11k 0,870 vs 0,858 (lebih baik
  semua), tetapi sensitivitas Icentia11k 0,843 vs 0,871 (lebih buruk). Sinus takikardia asli ≥ 120 bpm 0,46% [0,1–0,9] vs 98,4%.
- **Yang masih lemah:** alarm pada window non-AF ≥ 120 bpm di CPSC (label hanya "bukan AF", bisa termasuk aritmia lain) masih 25% (v3.1 94%); satu subjek
  (61) kehilangan sensitivitas 0,33; AFL hanya 144 window dari 2 subjek (D-1 92% vs v3.1 98% teralarm).
- **Pelajaran data:** label CPSC 2021 hanya menandai awal (`(AFIB`) dan akhir (`(N`) episode AF; rekaman non-AF tanpa label ritme. Aturan
  v15 (`segmen_ritme`) salah untuk kelas Normal di CPSC; jumlah AFL v15 tidak terpengaruh.